# ML 학습 데이터 0단계 묶음 만들기 (Colab)

저장소를 받아 **API를 부르지 않고** 지금 있는 템플릿(LLM 245개, 손 160개)과 privesc-llm-data로 ML 학습 데이터를 빌드한다. 결과는 바이트 단위로 재현되므로, 마지막 단계에서 저장소에 커밋된 기대 sha256(`metric-detection/templates/ml_train/handoff_step0.json`)과 대조한다.

순서: ① 설정 → ② 레포 받기 → ③ 패키지 → ④ privesc 받기 → ⑤ 템플릿 빌드·검증 → ⑥ privesc 빌드·검증 → ⑦ 내보내기 → ⑧ sha256 대조 → ⑨ 읽어 보기·Drive 저장.

각 단계는 `metric-detection/scripts/build_ml_handoff.sh`의 같은 이름 단계를 부른다. 셸에서는 `bash metric-detection/scripts/build_ml_handoff.sh` 한 줄로 전부 돈다. GPU는 필요 없다(CPU 런타임으로 약 2분).

**만든 파일에는 실제 형식의 가짜 비밀값과 privesc의 SSH 키가 들어 있다.** git에 올리거나 공유·공개 폴더에 두지 않는다.

## ① 설정

In [ ]:
# 받을 브랜치. handoff_step0.json의 sha256은 이 브랜치의 커밋 기준이다
BRANCH = "ml-train-pilot"
REPO_URL = "https://github.com/capstone-privai/promptguard-data.git"

# 결과를 압축해 복사할 Drive 폴더. None이면 복사하지 않는다 (비밀값 형식이 들어 있다. 공유 폴더에 두지 않는다)
OUT_DRIVE = None  # 예: "/content/drive/MyDrive/promptguard/ml_handoff"

REPO = "/content/promptguard-data"
PRIVESC_DIR = "/content/privesc-llm-data"

## ② 레포 받기

In [ ]:
import os, subprocess

def sh(cmd, check=True):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=check)

if not os.path.isdir(REPO):
    sh(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}")
else:
    sh(f"git -C {REPO} fetch --depth 1 origin {BRANCH} && git -C {REPO} checkout -q FETCH_HEAD")
sh(f"git -C {REPO} log -1 --oneline")
os.environ["PRIVESC_DIR"] = PRIVESC_DIR
STEP = f"bash {REPO}/metric-detection/scripts/build_ml_handoff.sh"

## ③ 패키지

`datasketch`는 내보내기의 중복 검사(MinHash LSH)에 쓰고, 결과가 늘 같도록 2.0.0으로 고정한다. `hf` CLI는 privesc를 받을 때 쓴다. Python은 3.11 이상이어야 한다(`tomllib`).

In [ ]:
sh("pip install -q datasketch==2.0.0 huggingface_hub")
import sys; print(sys.version)

## ④ privesc-llm-data 받기

[sailab-vienna/privesc-llm-data](https://huggingface.co/datasets/sailab-vienna/privesc-llm-data)를 고정 리비전 `39a9d2ff…`로 받는다(약 74MB, MIT). 이미 있으면 건너뛴다.

In [ ]:
sh(f"{STEP} download")

## ⑤ 템플릿 빌드·검증

`v1.jsonl`(LLM 245개 × 변형 3, 분포 이동 `o`는 2 → 730세션)과 `hand_v1.jsonl`(160개 × 5 → 800세션)을 `metric-detection/ml_build/`에 빌드하고, 각각 `verify`(LABELING 규칙, 자리, split 누수, 바이트 재생성)를 통과해야 한다. `raw/v1_local.jsonl`(Twilio SID 템플릿)은 git에 없어 빠진다.

In [ ]:
sh(f"{STEP} build")

## ⑥ privesc 빌드·검증

합성(손 + LLM) train 정답 수를 세어 privesc가 train 정답의 30% 이하가 되도록 시나리오를 돌아가며 표본을 뽑는다. `password_reuse`, `ssh_key_reuse`는 test_ood로 모두 들어간다. tool_input의 비밀은 `[PASSWORD_1]` 같은 가림 표시로 바뀐다.

In [ ]:
sh(f"{STEP} privesc")

## ⑦ 내보내기

train·val은 `metric-detection/ml_handoff/`로, test·test_ood는 `data_test/`·`data_answer/`의 `sessions_mleval_*`로 쓴다. 쓰기 전에 그룹 겹침, 값 누출, tool_input 정답, 분포 이동 특징, 거의 같은 문맥(중복)을 검사하고 하나라도 걸리면 아무것도 쓰지 않는다.

In [ ]:
sh(f"{STEP} export")

## ⑧ sha256 대조

데이터 파일 18개의 sha256을 `handoff_step0.json`과 비교한다. `OK`가 나와야 커밋된 것과 같은 데이터다. 이 단계가 `ml_handoff/README.md`(수, 비율, 한계, sha256)도 쓴다.

In [ ]:
sh(f"{STEP} check")

## ⑨ 읽어 보기 · Drive 저장

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{REPO}/metric-detection/ml_handoff/README.md", encoding="utf-8").read()))

In [ ]:
import json
md_dir = f"{REPO}/metric-detection"
def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]
train = read_jsonl(f"{md_dir}/ml_handoff/sessions_train.jsonl")
gold = read_jsonl(f"{md_dir}/ml_handoff/gold_train.jsonl")
s = train[0]
print(s["session_id"], s["meta"]["split"], s["meta"]["group_id"], [i["channel"] for i in s["items"]])
g = next(g for g in gold if g["session_id"] == s["session_id"])
item = s["items"][g["item_id"]]["text"]
print(g["span"]["type"], "length", g["span"]["end"] - g["span"]["start"])  # 값 자체는 출력하지 않는다

In [ ]:
if OUT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(OUT_DRIVE, exist_ok=True)
    sh(f"tar -czf {OUT_DRIVE}/ml_handoff_step0.tar.gz -C {md_dir} ml_handoff data_test data_answer "
       f"$(cd {md_dir} && ls labels_mleval_*.jsonl) manifest_mleval.json")
    sh(f"ls -l {OUT_DRIVE}")